In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import pandas as pd

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import string

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

In [2]:
df = pd.read_csv('/kaggle/input/ai-vs-human-text/AI_Human.csv')

print(df.head())
print(df.info())
print(df['generated'].value_counts())

                                                text  generated
0  Cars. Cars have been around since they became ...        0.0
1  Transportation is a large necessity in most co...        0.0
2  "America's love affair with it's vehicles seem...        0.0
3  How often do you ride in a car? Do you drive a...        0.0
4  Cars are a wonderful thing. They are perhaps o...        0.0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 487235 entries, 0 to 487234
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   text       487235 non-null  object 
 1   generated  487235 non-null  float64
dtypes: float64(1), object(1)
memory usage: 7.4+ MB
None
generated
0.0    305797
1.0    181438
Name: count, dtype: int64


In [3]:
def string_cleaner(text_string):
    text_string = text_string.translate(str.maketrans('','', string.punctuation))
    text_string = text_string.lower()
    return text_string

df['text_cleaned'] = df['text'].apply(string_cleaner)
print(df[['text', 'text_cleaned']].head())

                                                text  \
0  Cars. Cars have been around since they became ...   
1  Transportation is a large necessity in most co...   
2  "America's love affair with it's vehicles seem...   
3  How often do you ride in a car? Do you drive a...   
4  Cars are a wonderful thing. They are perhaps o...   

                                        text_cleaned  
0  cars cars have been around since they became f...  
1  transportation is a large necessity in most co...  
2  americas love affair with its vehicles seems t...  
3  how often do you ride in a car do you drive a ...  
4  cars are a wonderful thing they are perhaps on...  


In [4]:
try:
    stop_words_english = set(stopwords.words('english'))
    print(stop_words_english)
except LookupError:
    nltk.download('stopwords')
    stop_words_english = set(stopwords.words('english'))
    print(stop_words_english)

def stop_words_cleaner(text_string):
    if isinstance(text_string, str):
        words = text_string.split()
        filtered_words = [word for word in words if word not in stop_words_english]
        return " ".join(filtered_words)
    return " "

df['text_stop_words_cleaned'] = df['text_cleaned'].apply(stop_words_cleaner)
print(df[['text', 'text_cleaned', 'text_stop_words_cleaned']].head())

{'should', 'their', 'with', 'by', 'most', "he'd", 'him', 'having', 'as', 'few', "i'm", "you'd", 'for', 'on', "hasn't", "don't", 'over', 'were', 'while', 'what', 'nor', 'are', 'same', 'some', 'doing', 'but', 've', 'below', 'can', 'have', 'them', "she's", 'both', 'myself', 'does', 'how', 'so', "wouldn't", 'll', 'mustn', 'am', "it's", "they've", 'did', 'was', 'you', 'when', 'ours', 'about', 'now', 'hadn', 'between', 'been', 'they', 'we', 'shan', 'd', 'themselves', "mustn't", 'her', "they'll", 'above', 'm', 'ma', 'once', 'during', "they're", "shan't", 'very', "we'll", 'will', 'don', 'hasn', 'why', 'wouldn', 'an', 'yours', 'other', 'if', 'me', 'y', 'had', "he'll", "mightn't", 'up', 'more', 'yourself', 'there', 'then', 're', 'weren', "wasn't", 'and', 'its', 'after', 'here', 'i', 'our', 'the', 'against', 'needn', 'until', 'own', 'to', 'she', 'be', 'shouldn', "it'd", "she'd", 'whom', 'mightn', 'of', 'hers', 'doesn', "couldn't", "aren't", "she'll", "i've", 'further', 'won', 'too', 'these', "i'd

In [5]:
try:
    lemmatizer = WordNetLemmatizer()
    lemmatizer.lemmatize("running")
except LookupError:
    nltk.download('wordnet')
    lemmatizer = WordNetLemmatizer()

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text_string):
    if isinstance(text_string, str):
        words = text_string.split()
        lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
        return " ".join(lemmatized_words)
    return ""  

df['text_lemmatized'] = df['text_stop_words_cleaned'].apply(lemmatize_text)
print(df[['text', 'text_cleaned', 'text_stop_words_cleaned', 'text_lemmatized']].head()) 

                                                text  \
0  Cars. Cars have been around since they became ...   
1  Transportation is a large necessity in most co...   
2  "America's love affair with it's vehicles seem...   
3  How often do you ride in a car? Do you drive a...   
4  Cars are a wonderful thing. They are perhaps o...   

                                        text_cleaned  \
0  cars cars have been around since they became f...   
1  transportation is a large necessity in most co...   
2  americas love affair with its vehicles seems t...   
3  how often do you ride in a car do you drive a ...   
4  cars are a wonderful thing they are perhaps on...   

                             text_stop_words_cleaned  \
0  cars cars around since became famous 1900s hen...   
1  transportation large necessity countries world...   
2  americas love affair vehicles seems cooling sa...   
3  often ride car drive one motor vehicle work st...   
4  cars wonderful thing perhaps one worlds gre

In [6]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text_lemmatized'],
    df['generated'],
    test_size = 0.2,
    random_state = 42
)

tfidf_vectorizer = TfidfVectorizer()
train_tfidf_vectors = tfidf_vectorizer.fit_transform(train_texts)
test_tfidf_vectors = tfidf_vectorizer.transform(test_texts)

print("Train TF-IDF vektors shape:", train_tfidf_vectors.shape)
print("Test TF-IDF vektors shape:", test_tfidf_vectors.shape)

Train TF-IDF vektors shape: (389788, 251114)
Test TF-IDF vektors shape: (97447, 251114)


In [7]:
logistic_regression_model = LogisticRegression(solver = 'liblinear', random_state = 42)
naive_bayes_model = MultinomialNB()
svm_model = LinearSVC(random_state = 42)

logistic_regression_model.fit(train_tfidf_vectors, train_labels)
naive_bayes_model.fit(train_tfidf_vectors, train_labels)
svm_model.fit(train_tfidf_vectors, train_labels)

def evaluate_model(model, test_vectors, test_labels, model_name):
    pred = model.predict(test_vectors)
    accuracy = accuracy_score(test_labels, pred)
    report = classification_report(test_labels, pred)
    print(f'------- {model_name} Evaluation -------')
    print(f'Accuracy: {accuracy:.4f}')
    print('Classification Report:\n', report)

evaluate_model(logistic_regression_model, test_tfidf_vectors, test_labels, 'Logistic Regression')
evaluate_model(naive_bayes_model, test_tfidf_vectors, test_labels, 'Naive Bayes')
evaluate_model(svm_model, test_tfidf_vectors, test_labels, 'LinearSVC')

------- Logistic Regression Evaluation -------
Accuracy: 0.9938
Classification Report:
               precision    recall  f1-score   support

         0.0       0.99      1.00      1.00     61112
         1.0       1.00      0.99      0.99     36335

    accuracy                           0.99     97447
   macro avg       0.99      0.99      0.99     97447
weighted avg       0.99      0.99      0.99     97447

------- Naive Bayes Evaluation -------
Accuracy: 0.9512
Classification Report:
               precision    recall  f1-score   support

         0.0       0.94      0.99      0.96     61112
         1.0       0.98      0.89      0.93     36335

    accuracy                           0.95     97447
   macro avg       0.96      0.94      0.95     97447
weighted avg       0.95      0.95      0.95     97447

------- LinearSVC Evaluation -------
Accuracy: 0.9977
Classification Report:
               precision    recall  f1-score   support

         0.0       1.00      1.00      1.00  

In [8]:
def objective_logistic_regression(trial):
    C = trial.suggest_float('C', 1e-5, 100, log=True)
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])
    solver = 'liblinear'

    model = LogisticRegression(C=C, penalty=penalty, solver=solver, random_state=42)
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    accuracy_scores = []

    for fold, (train_index, val_index) in enumerate(kf.split(train_tfidf_vectors,train_labels)):
        X_train, X_val = train_tfidf_vectors[train_index], train_tfidf_vectors[val_index]
        y_train, y_val = train_labels.iloc[train_index], train_labels.iloc[val_index]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        accuracy = accuracy_score(y_val, y_pred)
        accuracy_scores.append(accuracy)
    
    return sum(accuracy_scores) / len(accuracy_scores)

study_logistic_regression = optuna.create_study(direction='maximize')
study_logistic_regression.optimize(objective_logistic_regression, n_trials=10)

print('Logistic Regression Optuna Results:\n')
print('Best trials: ', study_logistic_regression.best_trial)

print('\nBest parameters:')
for key, value, in study_logistic_regression.best_trial.params.items():
    print(f'{key}: {value}')

print(f'\nBest accuracy score: {study_logistic_regression.best_value:.4f}')

Logistic Regression Optuna Results:

Best trials:  FrozenTrial(number=9, state=1, values=[0.9975165979502686], datetime_start=datetime.datetime(2025, 5, 2, 10, 20, 21, 596926), datetime_complete=datetime.datetime(2025, 5, 2, 10, 23, 44, 805455), params={'C': 16.417476902961806, 'penalty': 'l2'}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'C': FloatDistribution(high=100.0, log=True, low=1e-05, step=None), 'penalty': CategoricalDistribution(choices=('l1', 'l2'))}, trial_id=9, value=None)

Best parameters:
C: 16.417476902961806
penalty: l2

Best accuracy score: 0.9975


In [9]:
def objective_svm(trial):
    C = trial.suggest_float('C', 1e-5, 100, log=True)
    loss = trial.suggest_categorical('loss', ['hinge', 'squared_hinge'])
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])
    dual = False  

    # Uyumsuz kombinasyonu engelle
    if penalty == 'l1' and loss == 'hinge':
        trial.set_user_attr('incompatible_combination', True)
        raise optuna.exceptions.TrialPruned()

    model = LinearSVC(C=C, loss=loss, penalty=penalty, dual=dual, random_state=42)
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    accuracy_scores = []

    for fold, (train_index, val_index) in enumerate(kf.split(train_tfidf_vectors, train_labels)):
        X_train, X_val = train_tfidf_vectors[train_index], train_tfidf_vectors[val_index]
        y_train, y_val = train_labels.iloc[train_index], train_labels.iloc[val_index]

        try:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            accuracy = accuracy_score(y_val, y_pred)
            accuracy_scores.append(accuracy)
        except ValueError as e:
            trial.set_user_attr('value_error', str(e))
            raise optuna.exceptions.TrialPruned()

    return sum(accuracy_scores) / len(accuracy_scores)

study_svm = optuna.create_study(direction='maximize')
study_svm.optimize(objective_svm, n_trials=10) 

print('\nSVM Optuna Results:\n')
print('Best trials:', study_svm.best_trial)

print('\nBest parameters:')
for key, value in study_svm.best_trial.params.items():
    print(f'{key}: {value}')

print(f'\nBest accuracy score: {study_svm.best_value:.4f}')


SVM Optuna Results:

Best trials: FrozenTrial(number=7, state=1, values=[0.9984222183690434], datetime_start=datetime.datetime(2025, 5, 2, 10, 30, 37, 116487), datetime_complete=datetime.datetime(2025, 5, 2, 10, 31, 47, 178320), params={'C': 21.314239654362126, 'loss': 'squared_hinge', 'penalty': 'l2'}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'C': FloatDistribution(high=100.0, log=True, low=1e-05, step=None), 'loss': CategoricalDistribution(choices=('hinge', 'squared_hinge')), 'penalty': CategoricalDistribution(choices=('l1', 'l2'))}, trial_id=7, value=None)

Best parameters:
C: 21.314239654362126
loss: squared_hinge
penalty: l2

Best accuracy score: 0.9984
